# Notebook 02: EDA – Parking Violations
**Project:** Parking-Induced Congestion AI System

This notebook:
- Loads `cleaned_violations.csv`
- Filters parking-related violations
- Generates EDA charts (top violations, stations, locations, hourly/weekday/monthly trends)
- Saves `parking_violations.csv` and all charts

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving files
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries imported.')

Libraries imported.


## 2. Define Paths

In [2]:
CLEANED_CSV       = '../data/processed/cleaned_violations.csv'
PARKING_CSV       = '../data/processed/parking_violations.csv'
CHARTS_DIR        = '../data/processed/charts/'

os.makedirs(CHARTS_DIR, exist_ok=True)
print('Paths ready.')

Paths ready.


## 3. Load Cleaned Data

In [3]:
df = pd.read_csv(CLEANED_CSV, low_memory=False)

# Parse datetime if present
if 'datetime' in df.columns:
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

print(f'Loaded cleaned data: {df.shape}')
display(df.head())

Loaded cleaned data: (298450, 31)


,id,latitude,longitude,location_or_junction,vehicle_type,vehicle_type.1,description,violation_type,offence_code,created_datetime,...,updated_vehicle_type,validation_status,validation_timestamp,datetime,hour,day,month,day_of_week,weekday_name,is_weekend
0,FKID000000,12.925557,77.618665,"18Th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,NaN,"[""Wrong Parking"",""Parking Near Road Crossing""]","[112,104]",2023-11-20 00:28:46+00,...,MAXI-CAB,approved,2023-11-30 03:08:24.818+00,2023-11-20 00:28:46+00:00,0.0,20.0,11.0,0.0,Monday,0
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,NaN,"[""No Parking""]",[113],2023-11-24 22:46:46+00,...,NaN,NaN,NaN,2023-11-24 22:46:46+00:00,22.0,24.0,11.0,4.0,Friday,0
2,FKID000002,12.925449,77.618504,"Koramangala 2Nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,NaN,"[""Wrong Parking"",""Parking In A Main Road""]","[112,107]",2023-11-20 00:27:46+00,...,MAXI-CAB,approved,2023-11-30 03:08:56.998+00,2023-11-20 00:27:46+00:00,0.0,20.0,11.0,0.0,Monday,0
3,FKID000003,12.956521,77.518618,"6Th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,NaN,"[""No Parking""]",[113],2023-11-16 06:47:46+00,...,SCOOTER,approved,2023-11-18 23:35:12.428+00,2023-11-16 06:47:46+00:00,6.0,16.0,11.0,3.0,Thursday,0
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,NaN,"[""No Parking""]",[113],2023-11-22 04:56:46+00,...,TANKER,approved,2023-11-30 03:11:32.796+00,2023-11-22 04:56:46+00:00,4.0,22.0,11.0,2.0,Wednesday,0


## 4. Confirm Canonical Columns

In [4]:
CANONICAL_COLS = ['latitude','longitude','datetime','violation_type','police_station',
                  'location_or_junction','vehicle_type','hour','day','month',
                  'day_of_week','weekday_name','is_weekend']

print('=== Canonical Column Status ===')
for col in CANONICAL_COLS:
    status = 'PRESENT' if col in df.columns else 'MISSING'
    print(f'  {col:30s} : {status}')

=== Canonical Column Status ===
  latitude                       : PRESENT
  longitude                      : PRESENT
  datetime                       : PRESENT
  violation_type                 : PRESENT
  police_station                 : PRESENT
  location_or_junction           : PRESENT
  vehicle_type                   : PRESENT
  hour                           : PRESENT
  day                            : PRESENT
  month                          : PRESENT
  day_of_week                    : PRESENT
  weekday_name                   : PRESENT
  is_weekend                     : PRESENT


## 5. Filter Parking-Related Violations

In [5]:
PARKING_KEYWORDS = [
    'parking', 'wrong parking', 'no parking', 'double parking',
    'footpath parking', 'main road parking', 'parked',
    'obstruction', 'no standing'
]

def is_parking_violation(text):
    """Return True if the violation text matches any parking keyword."""
    if pd.isna(text):
        return False
    text_lower = str(text).lower()
    return any(kw in text_lower for kw in PARKING_KEYWORDS)

if 'violation_type' in df.columns:
    parking_mask = df['violation_type'].apply(is_parking_violation)
    parking_df   = df[parking_mask].copy()
    print(f'Total records        : {len(df):,}')
    print(f'Parking-related      : {len(parking_df):,}')
    print(f'Parking percentage   : {len(parking_df)/len(df)*100:.1f}%')
    if len(parking_df) == 0:
        print('WARNING: No parking-related records found using keywords. Using full dataset as fallback.')
        parking_df = df.copy()
else:
    print('WARNING: violation_type column not found. Using full dataset as parking_df.')
    parking_df = df.copy()

display(parking_df['violation_type'].value_counts().head(10) if 'violation_type' in parking_df.columns else 'No violation_type column')

Total records        : 298,450
Parking-related      : 298,450
Parking percentage   : 100.0%


violation_type
["Wrong Parking"]                             138764
["No Parking"]                                119576
["Parking In A Main Road","Wrong Parking"]      9472
["Parking In A Main Road","No Parking"]         4818
["Wrong Parking","Defective Number Plate"]      3317
["No Parking","Parking In A Main Road"]         2449
["No Parking","Defective Number Plate"]         2380
["Wrong Parking","Parking In A Main Road"]      1955
["Parking On Footpath","Wrong Parking"]         1190
["No Parking","Wrong Parking"]                   891
Name: count, dtype: int64

## 6. Save Parking Violations CSV

In [6]:
parking_df.to_csv(PARKING_CSV, index=False)
print(f'Parking violations saved to: {PARKING_CSV}')
print(f'Shape: {parking_df.shape}')

Parking violations saved to: ../data/processed/parking_violations.csv
Shape: (298450, 31)


## 7. Helper – Save Chart

In [7]:
def save_chart(filename):
    path = os.path.join(CHARTS_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Chart saved: {path}')

## 8. Chart 1 – Top Violation Types

In [8]:
if 'violation_type' in parking_df.columns:
    top_viol = parking_df['violation_type'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(x=top_viol.values, y=top_viol.index, palette='Blues_r', ax=ax)
    ax.set_title('Top 15 Parking Violation Types', fontsize=14, fontweight='bold')
    ax.set_xlabel('Count')
    ax.set_ylabel('Violation Type')
    for i, v in enumerate(top_viol.values):
        ax.text(v + max(top_viol.values)*0.005, i, f'{v:,}', va='center', fontsize=9)
    save_chart('top_violation_types.png')
    display(top_viol)
else:
    print('Skipping: violation_type column not available.')

Chart saved: ../data/processed/charts/top_violation_types.png


violation_type
["Wrong Parking"]                                          138764
["No Parking"]                                             119576
["Parking In A Main Road","Wrong Parking"]                   9472
["Parking In A Main Road","No Parking"]                      4818
["Wrong Parking","Defective Number Plate"]                   3317
["No Parking","Parking In A Main Road"]                      2449
["No Parking","Defective Number Plate"]                      2380
["Wrong Parking","Parking In A Main Road"]                   1955
["Parking On Footpath","Wrong Parking"]                      1190
["No Parking","Wrong Parking"]                                891
["Parking In A Main Road","Wrong Parking","No Parking"]       865
["Wrong Parking","No Parking"]                                827
["Parking On Footpath","No Parking"]                          682
["No Parking","Wrong Parking","Parking In A Main Road"]       675
["Wrong Parking","Parking On Footpath"]                      

## 9. Chart 2 – Top Police Stations

In [9]:
if 'police_station' in parking_df.columns:
    top_ps = parking_df['police_station'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(x=top_ps.values, y=top_ps.index, palette='Oranges_r', ax=ax)
    ax.set_title('Top 15 Police Stations by Parking Violations', fontsize=14, fontweight='bold')
    ax.set_xlabel('Count')
    ax.set_ylabel('Police Station')
    save_chart('top_police_stations.png')
    display(top_ps)
else:
    print('Skipping: police_station column not available.')

Chart saved: ../data/processed/charts/top_police_stations.png


police_station
Upparpet             34468
Shivajinagar         28044
Malleshwaram         22200
Hal Old Airport      20819
City Market          17646
Vijayanagara         14652
Rajajinagar          10998
Kodigehalli          10916
Magadi Road           8558
Jeevanbheemanagar     6736
K.R. Pura             6546
Halasuru Gate         6294
Mahadevapura          6187
Chikkajala            5834
Hsr Layout            5018
Name: count, dtype: int64

## 10. Chart 3 – Top Junctions / Locations

In [10]:
if 'location_or_junction' in parking_df.columns and parking_df['location_or_junction'].notna().sum() > 0:
    top_loc = parking_df['location_or_junction'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(x=top_loc.values, y=top_loc.index, palette='Greens_r', ax=ax)
    ax.set_title('Top 15 Locations / Junctions by Parking Violations', fontsize=14, fontweight='bold')
    ax.set_xlabel('Count')
    ax.set_ylabel('Location / Junction')
    save_chart('top_locations.png')
    display(top_loc)
else:
    print('Skipping: location_or_junction column not available or empty.')

Chart saved: ../data/processed/charts/top_locations.png


location_or_junction
Unnamed Road, Begur Chikkanahalli, Bengaluru, Karnataka. Pin-562149 (India)                                                4090
Kamaraj Road, Sri Nagamma Devi Circle, Sivanchetti Gardens, Bengaluru, Karnataka. Pin-560042 (India)                       3999
New Horizon College Road, New Horizon College Of Engineering, Kadubisanahalli, Bengaluru, Karnataka. Pin-560103 (India)    3785
Mbt Road, Devasandra Junction, Kr Puram, Bengaluru, Karnataka. Pin-560036 (India)                                          3027
Dispensary Road, Tasker Town, Shivaji Nagar, Bengaluru, Karnataka. Pin-560001 (India)                                      2670
Bellary Road, Vinayaka Nagar, Hebbal, Bengaluru, Karnataka. Pin-560024 (India)                                             2639
5Th Main Road, Kempe Gowda Circle, Gandhi Nagar, Bengaluru, Karnataka. Pin-560009 (India)                                  2604
Main Guard Cross Road, Tasker Town, Shivaji Nagar, Bengaluru, Karnataka. Pin-560001

## 11. Chart 4 – Hourly Trend

In [11]:
if 'hour' in parking_df.columns and parking_df['hour'].notna().sum() > 0:
    hourly = parking_df['hour'].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.fill_between(hourly.index, hourly.values, alpha=0.4, color='steelblue')
    ax.plot(hourly.index, hourly.values, marker='o', color='steelblue', linewidth=2)
    ax.axvspan(8, 11, alpha=0.15, color='red', label='Morning Peak (8-11)')
    ax.axvspan(17, 21, alpha=0.15, color='orange', label='Evening Peak (17-21)')
    ax.set_title('Hourly Trend of Parking Violations', fontsize=14, fontweight='bold')
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Number of Violations')
    ax.set_xticks(range(0, 24))
    ax.legend()
    save_chart('hourly_trend.png')
    print(f'Busiest hour: {hourly.idxmax()} with {hourly.max():,} violations')
else:
    print('Skipping: hour column not available.')

Chart saved: ../data/processed/charts/hourly_trend.png
Busiest hour: 5.0 with 34,085 violations


## 12. Chart 5 – Weekday Trend

In [12]:
if 'weekday_name' in parking_df.columns and parking_df['weekday_name'].notna().sum() > 0:
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    weekday_counts = parking_df['weekday_name'].value_counts().reindex(day_order, fill_value=0)
    colors = ['#e74c3c' if d in ['Saturday','Sunday'] else '#3498db' for d in day_order]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(weekday_counts.index, weekday_counts.values, color=colors)
    ax.set_title('Weekday Trend of Parking Violations', fontsize=14, fontweight='bold')
    ax.set_xlabel('Day of Week')
    ax.set_ylabel('Number of Violations')
    plt.xticks(rotation=30)
    save_chart('weekday_trend.png')
    print(f'Busiest weekday: {weekday_counts.idxmax()} with {weekday_counts.max():,} violations')
else:
    print('Skipping: weekday_name column not available.')

Chart saved: ../data/processed/charts/weekday_trend.png
Busiest weekday: Sunday with 46,863 violations


## 13. Chart 6 – Monthly Trend

In [13]:
if 'month' in parking_df.columns and parking_df['month'].notna().sum() > 0:
    month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                   7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
    monthly = parking_df['month'].value_counts().sort_index()
    monthly.index = [month_names.get(m, str(m)) for m in monthly.index]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(monthly.index, monthly.values, color='teal', alpha=0.8)
    ax.set_title('Monthly Trend of Parking Violations', fontsize=14, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Number of Violations')
    save_chart('monthly_trend.png')
    display(monthly)
else:
    print('Skipping: month column not available.')

Chart saved: ../data/processed/charts/monthly_trend.png


Jan    65813
Feb    54650
Mar    55229
Apr    15082
Nov    44117
Dec    63554
Name: count, dtype: int64

## 14. Chart 7 – Vehicle Type Distribution (if available)

In [14]:
if 'vehicle_type' in parking_df.columns and parking_df['vehicle_type'].notna().sum() > 0:
    top_veh = parking_df['vehicle_type'].value_counts().head(12)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=top_veh.index, y=top_veh.values, palette='Set2', ax=ax)
    ax.set_title('Vehicle Type Distribution in Parking Violations', fontsize=14, fontweight='bold')
    ax.set_xlabel('Vehicle Type')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    save_chart('vehicle_type_distribution.png')
    display(top_veh)
else:
    print('Skipping: vehicle_type column not available or empty.')

Chart saved: ../data/processed/charts/vehicle_type_distribution.png


vehicle_type
FKN00GL4424     55
FKN00GL3514     42
FKN00GL9771     41
FKN00GL17863    41
FKN00GL2906     35
FKN00GL15265    34
FKN00GL14092    34
FKN00GL1875     30
FKN00GL19337    30
FKN00GL9852     29
FKN00GL1118     28
FKN00GL15753    28
Name: count, dtype: int64

## 15. Key EDA Insights Summary

In [15]:
total_records   = len(df)
parking_records = len(parking_df)
pct_parking     = parking_records / total_records * 100 if total_records > 0 else 0

busiest_hour    = int(parking_df['hour'].mode()[0])    if 'hour' in parking_df.columns and parking_df['hour'].notna().sum() > 0 else 'N/A'
busiest_weekday = parking_df['weekday_name'].mode()[0] if 'weekday_name' in parking_df.columns and parking_df['weekday_name'].notna().sum() > 0 else 'N/A'
top_ps          = parking_df['police_station'].mode()[0] if 'police_station' in parking_df.columns and parking_df['police_station'].notna().sum() > 0 else 'N/A'
top_loc         = parking_df['location_or_junction'].mode()[0] if 'location_or_junction' in parking_df.columns and parking_df['location_or_junction'].notna().sum() > 0 else 'N/A'

print('=' * 50)
print('KEY EDA INSIGHTS')
print('=' * 50)
print(f'  Total records              : {total_records:,}')
print(f'  Total parking-related      : {parking_records:,}')
print(f'  Percentage parking-related : {pct_parking:.1f}%')
print(f'  Busiest hour               : {busiest_hour}:00')
print(f'  Busiest weekday            : {busiest_weekday}')
print(f'  Top police station         : {top_ps}')
print(f'  Top location/junction      : {top_loc}')
print('=' * 50)

KEY EDA INSIGHTS
  Total records              : 298,450
  Total parking-related      : 298,450
  Percentage parking-related : 100.0%
  Busiest hour               : 5:00
  Busiest weekday            : Sunday
  Top police station         : Upparpet
  Top location/junction      : Unnamed Road, Begur Chikkanahalli, Bengaluru, Karnataka. Pin-562149 (India)
